# ForecastMaster R04.2 cycle feed
First import the R04.2 visual into the existing report and save it. Then Run all here. This updates six transport expressions in the existing semantic model. Source tables, partitions, calculations, report pages, native button, Functions and pipelines retain their existing definitions. A readable model backup is verified in ForecastMaster before the update. Company speed must be checked after installation.


In [ ]:
# Run after importing the R04.2 visual into the existing report and saving it.
"""Update only the existing report feed measures, using the operator's Fabric session."""
from __future__ import annotations

import base64
import copy
import hashlib
import json
import re
import time
from datetime import datetime, timezone

WORKSPACE = 'f5cb5742-9a7c-4337-a4c0-952706dddf0b'
MODEL = 'e7af2967-7725-4108-a170-9e490d4e35bb'
LAKEHOUSE = 'd6e33ebd-67b1-420a-8d84-c3a0d04d8b21'
ROOT = f'https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE}/semanticModels/{MODEL}'


def normalized(expression):
    if isinstance(expression, list):
        expression = '\n'.join(expression)
    # Ignore formatting only outside DAX quoted strings and table identifiers.
    return re.sub(r'"(?:""|[^"])*"|\'(?:\'\'|[^\'])*\'|\s+', lambda m: '' if m[0].isspace() else m[0], expression)


def unpack(definition):
    parts = definition['parts']
    if len({p['path'] for p in parts}) != len(parts):
        raise RuntimeError('Duplicate definition part paths; no update prepared.')
    found = [p for p in parts if p['path'] == 'model.bim']
    if len(found) != 1 or found[0].get('payloadType') != 'InlineBase64':
        raise RuntimeError('Expected the existing TMSL model.bim; no update prepared.')
    return found[0], json.loads(base64.b64decode(found[0]['payload'], validate=True))


def measures(model):
    tables = [t for t in model['model']['tables'] if t['name'] == 'fm_review_measures']
    if len(tables) != 1:
        raise RuntimeError('Expected the existing fm_review_measures table.')
    return {m['name']: m for m in tables[0]['measures']}


def prepare(definition, expected, updates):
    result = copy.deepcopy(definition)
    part, model = unpack(result)
    existing = measures(model)
    changed = []
    for name, update in updates.items():
        if name not in existing:
            raise RuntimeError(f'Missing existing measure {name}; no update prepared.')
        actual = normalized(existing[name]['expression'])
        if actual not in {normalized(expected[name]['expression']), normalized(update['expression'])}:
            raise RuntimeError(f'{name} differs from the qualified R04 feed. No update prepared.')
        if actual != normalized(update['expression']):
            existing[name]['expression'] = update['expression']
            changed.append(name)
    part['payload'] = base64.b64encode(json.dumps(model, ensure_ascii=False).encode()).decode()
    return result, changed


def unchanged_except_feed(before, after, updates):
    _, a = unpack(before)
    _, b = unpack(after)
    for value in [a, b]:
        for name in updates:
            measures(value)[name]['expression'] = '<qualified feed expression>'
    if a != b:
        raise RuntimeError('Model properties outside the feed measures differ.')
    def other(d):
        return {p['path']: p for p in d['parts'] if p['path'] != 'model.bim'}
    if other(before) != other(after):
        raise RuntimeError('Definition parts outside model.bim differ.')


class Fabric:
    def __init__(self, token, http, sleep=time.sleep, now=time.monotonic):
        self.http, self.sleep, self.now = http, sleep, now
        self.headers = {'Authorization': 'Bearer ' + token, 'Content-Type': 'application/json'}

    def call(self, method, url, body=None):
        # Never replay an ambiguous write after a timeout or transient response.
        response = self.http.request(method, url, headers=self.headers, json=body, timeout=90)
        response.raise_for_status()
        if response.status_code != 202:
            return response.json() if response.content else {}
        operation = response.headers.get('Location')
        if not operation or not operation.startswith('https://api.fabric.microsoft.com/v1/operations/'):
            raise RuntimeError('Missing Fabric operation URL; outcome requires inspection.')
        deadline = self.now() + 600
        while self.now() < deadline:
            self.sleep(min(max(int(response.headers.get('Retry-After', '3')), 1), 30))
            response = self.http.get(operation, headers=self.headers, timeout=90)
            response.raise_for_status()
            status = response.json().get('status')
            if status in ['Failed', 'Canceled', 'Cancelled']:
                raise RuntimeError('Fabric definition operation ' + str(response.json()))
            if status == 'Succeeded':
                result = self.http.get(operation.rstrip('/') + '/result', headers=self.headers, timeout=90)
                if result.status_code in [204, 404]:
                    return {}
                result.raise_for_status()
                return result.json() if result.content else {}
            print('Waiting for report feed update:', status)
        raise RuntimeError('Definition operation is still unconfirmed after 10 minutes. Do not repeat the write.')

    def read(self):
        return self.call('POST', ROOT + '/getDefinition?format=TMSL')['definition']


def install(api, fs, expected, updates):
    old = api.read()
    new, changed = prepare(old, expected, updates)
    unchanged_except_feed(old, new, updates)
    if not changed:
        return {'Status': 'ALREADY_INSTALLED', 'Release': 'R04.2', 'ChangedMeasures': []}
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    folder = f'abfss://{WORKSPACE}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE}/Files/ForecastMasterRefreshState/report-upgrades/R04.2/{stamp}'
    fs.mkdirs(folder)
    backup = json.dumps(old, ensure_ascii=False)
    fs.put(folder + '/model-before.json', backup, False)
    saved = fs.head(folder + '/model-before.json', len(backup.encode()) + 1024)
    if saved != backup:
        raise RuntimeError('Could not verify the readable model backup; no update submitted.')
    print('Backup:', folder + '/model-before.json')
    current = api.read()
    if unpack(current)[1] != unpack(old)[1] or {p['path']: p for p in current['parts'] if p['path'] != 'model.bim'} != {p['path']: p for p in old['parts'] if p['path'] != 'model.bim'}:
        raise RuntimeError('The model changed during preparation; no update submitted.')
    print('Updating only:', ', '.join(changed))
    api.call('POST', ROOT + '/updateDefinition', {'definition': new, 'options': {'allowPurgeData': False}})
    actual = api.read()
    unchanged_except_feed(old, actual, updates)
    for name, update in updates.items():
        if normalized(measures(unpack(actual)[1])[name]['expression']) != normalized(update['expression']):
            raise RuntimeError('The installed expression is unconfirmed: ' + name + '. Retain the backup and output.')
    receipt = {'Status': 'INSTALLED', 'Release': 'R04.2', 'ChangedMeasures': changed, 'Backup': folder + '/model-before.json', 'DataPurgeAllowed': False, 'ReportPagesAndButtons': 'Not modified by this installer', 'CompanyInteractionSpeed': 'NOT_YET_MEASURED', 'InstalledDefinitionSHA256': hashlib.sha256(json.dumps(actual, sort_keys=True).encode()).hexdigest()}
    fs.put(folder + '/receipt.json', json.dumps(receipt, indent=2), False)
    return receipt


def main(expected, updates):
    import requests
    import notebookutils
    api = Fabric(notebookutils.credentials.getToken('pbi'), requests)
    result = install(api, notebookutils.fs, expected, updates)
    print(json.dumps(result, indent=2))
    print('READY: R04.2 cycle feed verified. Reload the report with the R04.2 visual, select a demand cycle, then test the other filters.')
    return result

EXPECTED = json.loads('{"FM UI Numerator": {"expression": "VAR _scopeAllowed = CALCULATE(DISTINCTCOUNT(\'fm_review_reference\'[Country]), ALLSELECTED(\'fm_review_reference\')) = 1 && CALCULATE(DISTINCTCOUNT(\'fm_review_reference\'[SourceProfile]), ALLSELECTED(\'fm_review_reference\')) = 1 && CALCULATE(DISTINCTCOUNT(\'fm_review_measures\'[MetricKey]), ALLSELECTED(\'fm_review_measures\')) = 1\\nRETURN IF(_scopeAllowed,\\nVAR _id = SELECTEDVALUE(\'fm_review_files\'[SourceRevisionID])\\nVAR _month = SELECTEDVALUE(\'fm_review_axis\'[Month])\\nVAR _key = SELECTEDVALUE(\'fm_review_measures\'[NumeratorKey])\\nVAR _allowed = CONTAINS(VALUES(\'fm_review_reference\'[SourceRevisionID]), \'fm_review_reference\'[SourceRevisionID], _id)\\nRETURN IF(_allowed && NOT ISBLANK(_key) && HASONEVALUE(\'fm_review_measures\'[MetricKey]),\\n    CALCULATE(SUM(\'fm_review_facts\'[Amount]), TREATAS({_id}, \'fm_review_facts\'[SourceRevisionID]),\\n        TREATAS({_month}, \'fm_review_facts\'[Month]), TREATAS({_key}, \'fm_review_facts\'[ComponentKey])))\\n)", "formatString": "0.############", "displayFolder": "Forecast review\\\\Visual feed", "dynamic": false}, "FM UI Denominator": {"expression": "VAR _scopeAllowed = CALCULATE(DISTINCTCOUNT(\'fm_review_reference\'[Country]), ALLSELECTED(\'fm_review_reference\')) = 1 && CALCULATE(DISTINCTCOUNT(\'fm_review_reference\'[SourceProfile]), ALLSELECTED(\'fm_review_reference\')) = 1 && CALCULATE(DISTINCTCOUNT(\'fm_review_measures\'[MetricKey]), ALLSELECTED(\'fm_review_measures\')) = 1\\nRETURN IF(_scopeAllowed,\\nVAR _id = SELECTEDVALUE(\'fm_review_files\'[SourceRevisionID])\\nVAR _month = SELECTEDVALUE(\'fm_review_axis\'[Month])\\nVAR _key = SELECTEDVALUE(\'fm_review_measures\'[DenominatorKey])\\nVAR _allowed = CONTAINS(VALUES(\'fm_review_reference\'[SourceRevisionID]), \'fm_review_reference\'[SourceRevisionID], _id)\\nRETURN IF(_allowed && NOT ISBLANK(_key) && HASONEVALUE(\'fm_review_measures\'[MetricKey]),\\n    CALCULATE(SUM(\'fm_review_facts\'[Amount]), TREATAS({_id}, \'fm_review_facts\'[SourceRevisionID]),\\n        TREATAS({_month}, \'fm_review_facts\'[Month]), TREATAS({_key}, \'fm_review_facts\'[ComponentKey])))\\n)", "formatString": "0.############", "displayFolder": "Forecast review\\\\Visual feed", "dynamic": false}, "FM UI Numerator Missing": {"expression": "VAR _scopeAllowed = CALCULATE(DISTINCTCOUNT(\'fm_review_reference\'[Country]), ALLSELECTED(\'fm_review_reference\')) = 1 && CALCULATE(DISTINCTCOUNT(\'fm_review_reference\'[SourceProfile]), ALLSELECTED(\'fm_review_reference\')) = 1 && CALCULATE(DISTINCTCOUNT(\'fm_review_measures\'[MetricKey]), ALLSELECTED(\'fm_review_measures\')) = 1\\nRETURN IF(_scopeAllowed,\\nVAR _id = SELECTEDVALUE(\'fm_review_files\'[SourceRevisionID])\\nVAR _month = SELECTEDVALUE(\'fm_review_axis\'[Month])\\nVAR _key = SELECTEDVALUE(\'fm_review_measures\'[NumeratorKey])\\nVAR _allowed = CONTAINS(VALUES(\'fm_review_reference\'[SourceRevisionID]), \'fm_review_reference\'[SourceRevisionID], _id)\\nRETURN IF(_allowed && NOT ISBLANK(_key) && HASONEVALUE(\'fm_review_measures\'[MetricKey]),\\n    CALCULATE(SUM(\'fm_review_facts\'[MissingCells]), TREATAS({_id}, \'fm_review_facts\'[SourceRevisionID]),\\n        TREATAS({_month}, \'fm_review_facts\'[Month]), TREATAS({_key}, \'fm_review_facts\'[ComponentKey])))\\n)", "formatString": "0.############", "displayFolder": "Forecast review\\\\Visual feed", "dynamic": false}, "FM UI Denominator Missing": {"expression": "VAR _scopeAllowed = CALCULATE(DISTINCTCOUNT(\'fm_review_reference\'[Country]), ALLSELECTED(\'fm_review_reference\')) = 1 && CALCULATE(DISTINCTCOUNT(\'fm_review_reference\'[SourceProfile]), ALLSELECTED(\'fm_review_reference\')) = 1 && CALCULATE(DISTINCTCOUNT(\'fm_review_measures\'[MetricKey]), ALLSELECTED(\'fm_review_measures\')) = 1\\nRETURN IF(_scopeAllowed,\\nVAR _id = SELECTEDVALUE(\'fm_review_files\'[SourceRevisionID])\\nVAR _month = SELECTEDVALUE(\'fm_review_axis\'[Month])\\nVAR _key = SELECTEDVALUE(\'fm_review_measures\'[DenominatorKey])\\nVAR _allowed = CONTAINS(VALUES(\'fm_review_reference\'[SourceRevisionID]), \'fm_review_reference\'[SourceRevisionID], _id)\\nRETURN IF(_allowed && NOT ISBLANK(_key) && HASONEVALUE(\'fm_review_measures\'[MetricKey]),\\n    CALCULATE(SUM(\'fm_review_facts\'[MissingCells]), TREATAS({_id}, \'fm_review_facts\'[SourceRevisionID]),\\n        TREATAS({_month}, \'fm_review_facts\'[Month]), TREATAS({_key}, \'fm_review_facts\'[ComponentKey])))\\n)", "formatString": "0.############", "displayFolder": "Forecast review\\\\Visual feed", "dynamic": false}, "FM UI File": {"expression": "VAR _scopeAllowed = CONTAINS(VALUES(\'fm_review_reference\'[SourceRevisionID]), \'fm_review_reference\'[SourceRevisionID], MINX(ALLSELECTED(\'fm_review_reference\'), \'fm_review_reference\'[SourceRevisionID])) && SELECTEDVALUE(\'fm_review_measures\'[MetricKey]) = MINX(ALLSELECTED(\'fm_review_measures\'), \'fm_review_measures\'[MetricKey])\\nRETURN IF(_scopeAllowed,\\nVAR _month = SELECTEDVALUE(\'fm_review_axis\'[Month])\\nVAR _first = MINX(ALL(\'fm_review_axis\'), \'fm_review_axis\'[Month])\\nRETURN IF(_month = _first, \\"{\\" & \\"\\"\\"id\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(COALESCE(SELECTEDVALUE(\'fm_review_files\'[SourceRevisionID]), \\"\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"fullLabel\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(COALESCE(SELECTEDVALUE(\'fm_review_files\'[FileLabel]), \\"\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"filename\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(COALESCE(SELECTEDVALUE(\'fm_review_files\'[SourceFileName]), \\"\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"path\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(COALESCE(SELECTEDVALUE(\'fm_review_files\'[SourceRelativePath]), \\"\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"affiliate\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(COALESCE(SELECTEDVALUE(\'fm_review_files\'[Country]), \\"\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"cycle\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(COALESCE(SELECTEDVALUE(\'fm_review_files\'[DemandCycle]), \\"\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"driver\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(COALESCE(SELECTEDVALUE(\'fm_review_files\'[GrowthDriver]), \\"\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"profile\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(COALESCE(SELECTEDVALUE(\'fm_review_files\'[SourceProfile]), \\"\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"status\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(COALESCE(SELECTEDVALUE(\'fm_review_files\'[PublicationStatus]), \\"\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"ims\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(FORMAT(SELECTEDVALUE(\'fm_review_files\'[IMSActualsThrough]), \\"yyyy-MM\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"ef\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(FORMAT(SELECTEDVALUE(\'fm_review_files\'[ExFactoryActualsThrough]), \\"yyyy-MM\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\"}\\")\\n)", "formatString": "", "displayFolder": "Forecast review\\\\Visual feed", "dynamic": false}, "FM UI Catalog": {"expression": "VAR _scopeAllowed = CONTAINS(VALUES(\'fm_review_reference\'[SourceRevisionID]), \'fm_review_reference\'[SourceRevisionID], MINX(ALLSELECTED(\'fm_review_reference\'), \'fm_review_reference\'[SourceRevisionID])) && SELECTEDVALUE(\'fm_review_measures\'[MetricKey]) = MINX(ALLSELECTED(\'fm_review_measures\'), \'fm_review_measures\'[MetricKey])\\nRETURN IF(_scopeAllowed,\\nVAR _firstMonth = MINX(ALL(\'fm_review_axis\'), \'fm_review_axis\'[Month])\\nVAR _firstFile = MINX(ALL(\'fm_review_files\'), \'fm_review_files\'[SourceRevisionID])\\nVAR _first = SELECTEDVALUE(\'fm_review_axis\'[Month]) = _firstMonth && SELECTEDVALUE(\'fm_review_files\'[SourceRevisionID]) = _firstFile\\n\\nRETURN IF(_first, \\"[\\" & CONCATENATEX(ALL(\'fm_review_measures\'), \\"{\\" & \\"\\"\\"id\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(\'fm_review_measures\'[MetricKey], UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"family\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(\'fm_review_measures\'[Section], UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"label\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(\'fm_review_measures\'[Measure], UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"unit\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(\'fm_review_measures\'[Unit], UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"aggregation\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(\'fm_review_measures\'[Aggregation], UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"cutoffFamily\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(\'fm_review_measures\'[CutoffFamily], UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"numeratorKey\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(\'fm_review_measures\'[NumeratorKey], UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"denominatorKey\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(\'fm_review_measures\'[DenominatorKey], UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"availabilityIndex\\"\\":\\" & FORMAT(RANKX(ALL(\'fm_review_measures\'), \'fm_review_measures\'[MetricKey], , ASC, DENSE) - 1, \\"0\\") & \\"}\\", \\",\\", \'fm_review_measures\'[SortIndex], ASC) & \\"]\\")\\n)", "formatString": "", "displayFolder": "Forecast review\\\\Visual feed", "dynamic": false}, "FM UI Context": {"expression": "VAR _scopeAllowed = CONTAINS(VALUES(\'fm_review_reference\'[SourceRevisionID]), \'fm_review_reference\'[SourceRevisionID], MINX(ALLSELECTED(\'fm_review_reference\'), \'fm_review_reference\'[SourceRevisionID])) && SELECTEDVALUE(\'fm_review_measures\'[MetricKey]) = MINX(ALLSELECTED(\'fm_review_measures\'), \'fm_review_measures\'[MetricKey])\\nRETURN IF(_scopeAllowed,\\nVAR _firstMonth = MINX(ALL(\'fm_review_axis\'), \'fm_review_axis\'[Month])\\nVAR _firstFile = MINX(ALL(\'fm_review_files\'), \'fm_review_files\'[SourceRevisionID])\\nVAR _first = SELECTEDVALUE(\'fm_review_axis\'[Month]) = _firstMonth && SELECTEDVALUE(\'fm_review_files\'[SourceRevisionID]) = _firstFile\\n\\nRETURN IF(_first, \\"{\\" & \\"\\"\\"affiliate\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(CALCULATE(SELECTEDVALUE(\'fm_review_reference\'[Country], \\"\\"), ALLSELECTED(\'fm_review_reference\')), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"profile\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(CALCULATE(SELECTEDVALUE(\'fm_review_reference\'[SourceProfile], \\"\\"), ALLSELECTED(\'fm_review_reference\')), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"metric\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(CALCULATE(SELECTEDVALUE(\'fm_review_measures\'[MetricKey], \\"\\"), ALLSELECTED(\'fm_review_measures\')), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"channel\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(IF(ISFILTERED(\'fm_review_facts\'[Channel]), SELECTEDVALUE(\'fm_review_facts\'[Channel], \\"all\\"), \\"all\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"subchannel\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(IF(ISFILTERED(\'fm_review_facts\'[SubChannel]), SELECTEDVALUE(\'fm_review_facts\'[SubChannel], \\"all\\"), \\"all\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"sector\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(IF(ISFILTERED(\'fm_review_facts\'[Sector]), SELECTEDVALUE(\'fm_review_facts\'[Sector], \\"all\\"), \\"all\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"schema\\"\\":\\" & \\"4\\" & \\",\\" & \\"\\"\\"cycle\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(CALCULATE(SELECTEDVALUE(\'fm_review_reference\'[DemandCycle], \\"\\"), ALLSELECTED(\'fm_review_reference\')), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"driver\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(CALCULATE(SELECTEDVALUE(\'fm_review_reference\'[GrowthDriver], \\"\\"), ALLSELECTED(\'fm_review_reference\')), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"cycles\\"\\":\\" & \\"[\\" & CONCATENATEX(SUMMARIZE(ALLSELECTED(\'fm_review_reference\'), \'fm_review_reference\'[DemandCycle]), UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(\'fm_review_reference\'[DemandCycle], UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34), \\",\\", \'fm_review_reference\'[DemandCycle], ASC) & \\"]\\" & \\",\\" & \\"\\"\\"drivers\\"\\":\\" & \\"[\\" & CONCATENATEX(SUMMARIZE(ALLSELECTED(\'fm_review_reference\'), \'fm_review_reference\'[GrowthDriver]), UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(\'fm_review_reference\'[GrowthDriver], UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34), \\",\\", \'fm_review_reference\'[GrowthDriver], ASC) & \\"]\\" & \\"}\\")\\n)", "formatString": "", "displayFolder": "Forecast review\\\\Visual feed", "dynamic": false}, "FM UI Dimensions": {"expression": "VAR _scopeAllowed = CONTAINS(VALUES(\'fm_review_reference\'[SourceRevisionID]), \'fm_review_reference\'[SourceRevisionID], MINX(ALLSELECTED(\'fm_review_reference\'), \'fm_review_reference\'[SourceRevisionID])) && SELECTEDVALUE(\'fm_review_measures\'[MetricKey]) = MINX(ALLSELECTED(\'fm_review_measures\'), \'fm_review_measures\'[MetricKey])\\nRETURN IF(_scopeAllowed,\\nVAR _firstMonth = MINX(ALL(\'fm_review_axis\'), \'fm_review_axis\'[Month])\\nVAR _firstFile = MINX(ALL(\'fm_review_files\'), \'fm_review_files\'[SourceRevisionID])\\nVAR _first = SELECTEDVALUE(\'fm_review_axis\'[Month]) = _firstMonth && SELECTEDVALUE(\'fm_review_files\'[SourceRevisionID]) = _firstFile\\n\\nVAR _ids = DISTINCT(SELECTCOLUMNS(ALLSELECTED(\'fm_review_reference\'), \\"ID\\", \'fm_review_reference\'[SourceRevisionID]))\\nVAR _dimensions = CALCULATETABLE(SUMMARIZE(\'fm_review_facts\', \'fm_review_facts\'[Channel], \'fm_review_facts\'[SubChannel], \'fm_review_facts\'[Sector]),\\n    REMOVEFILTERS(\'fm_review_facts\'[Channel], \'fm_review_facts\'[SubChannel], \'fm_review_facts\'[Sector]), TREATAS(_ids, \'fm_review_facts\'[SourceRevisionID]))\\nVAR _rows = ADDCOLUMNS(_dimensions, \\"__metrics\\",\\n    CONCATENATEX(FILTER(ALL(\'fm_review_measures\'),\\n        VAR _numKey = \'fm_review_measures\'[NumeratorKey]\\n        VAR _denKey = \'fm_review_measures\'[DenominatorKey]\\n        VAR _reportedNumerator = CALCULATE(SUM(\'fm_review_facts\'[NumericCells]),\\n            TREATAS(_ids, \'fm_review_facts\'[SourceRevisionID]), TREATAS({_numKey}, \'fm_review_facts\'[ComponentKey]))\\n        VAR _reportedDenominator = CALCULATE(SUM(\'fm_review_facts\'[NumericCells]),\\n            TREATAS(_ids, \'fm_review_facts\'[SourceRevisionID]), TREATAS({_denKey}, \'fm_review_facts\'[ComponentKey]))\\n        RETURN _reportedNumerator > 0 && (\'fm_review_measures\'[Aggregation] <> \\"RATIO\\" || _reportedDenominator > 0)),\\n        FORMAT(RANKX(ALL(\'fm_review_measures\'), \'fm_review_measures\'[MetricKey], , ASC, DENSE) - 1, \\"0\\"), \\",\\"))\\nRETURN IF(_first, \\"[\\" & CONCATENATEX(_rows, \\"{\\" & \\"\\"\\"channel\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(\'fm_review_facts\'[Channel], UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"subchannel\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(\'fm_review_facts\'[SubChannel], UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"sector\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(\'fm_review_facts\'[Sector], UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"metrics\\"\\":\\" & \\"[\\" & [__metrics] & \\"]\\" & \\"}\\", \\",\\") & \\"]\\")\\n)", "formatString": "", "displayFolder": "Forecast review\\\\Visual feed", "dynamic": false}, "FM UI Bounds": {"expression": "VAR _scopeAllowed = CONTAINS(VALUES(\'fm_review_reference\'[SourceRevisionID]), \'fm_review_reference\'[SourceRevisionID], MINX(ALLSELECTED(\'fm_review_reference\'), \'fm_review_reference\'[SourceRevisionID])) && SELECTEDVALUE(\'fm_review_measures\'[MetricKey]) = MINX(ALLSELECTED(\'fm_review_measures\'), \'fm_review_measures\'[MetricKey])\\nRETURN IF(_scopeAllowed,\\nVAR _firstMonth = MINX(ALL(\'fm_review_axis\'), \'fm_review_axis\'[Month])\\nVAR _firstFile = MINX(ALL(\'fm_review_files\'), \'fm_review_files\'[SourceRevisionID])\\nVAR _first = SELECTEDVALUE(\'fm_review_axis\'[Month]) = _firstMonth && SELECTEDVALUE(\'fm_review_files\'[SourceRevisionID]) = _firstFile\\nRETURN IF(_first, \\"{\\" & \\"\\"\\"first\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(FORMAT(MINX(ALL(\'fm_review_axis\'), \'fm_review_axis\'[Month]), \\"yyyy-MM\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"last\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(FORMAT(MAXX(ALL(\'fm_review_axis\'), \'fm_review_axis\'[Month]), \\"yyyy-MM\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\"}\\")\\n)", "formatString": "", "displayFolder": "Forecast review\\\\Visual feed", "dynamic": false}}')
UPDATES = json.loads('{"FM UI Numerator": {"expression": "VAR _country = CALCULATE(SELECTEDVALUE(\'fm_review_reference\'[Country]), ALLSELECTED(\'fm_review_reference\'))\\nVAR _cycles = CALCULATETABLE(VALUES(\'fm_review_reference\'[DemandCycle]), ALLSELECTED(\'fm_review_reference\'))\\nVAR _selectedIDs = SELECTCOLUMNS(ALLSELECTED(\'fm_review_reference\'), \\"ID\\", \'fm_review_reference\'[SourceRevisionID])\\nVAR _expectedIDs = SELECTCOLUMNS(FILTER(ALL(\'fm_review_reference\'), \'fm_review_reference\'[Country] = _country && \'fm_review_reference\'[DemandCycle] IN _cycles), \\"ID\\", \'fm_review_reference\'[SourceRevisionID])\\nVAR _allMetrics = CALCULATE(DISTINCTCOUNT(\'fm_review_measures\'[MetricKey]), ALLSELECTED(\'fm_review_measures\')) = COUNTROWS(ALL(\'fm_review_measures\'))\\nVAR _cycleComplete = NOT ISBLANK(_country) && ISEMPTY(EXCEPT(_expectedIDs, _selectedIDs)) && _allMetrics\\nVAR _scopeAllowed = _cycleComplete\\nRETURN IF(_scopeAllowed,\\nVAR _id = SELECTEDVALUE(\'fm_review_files\'[SourceRevisionID])\\nVAR _month = SELECTEDVALUE(\'fm_review_axis\'[Month])\\nVAR _key = SELECTEDVALUE(\'fm_review_measures\'[NumeratorKey])\\nVAR _allowed = CONTAINS(VALUES(\'fm_review_reference\'[SourceRevisionID]), \'fm_review_reference\'[SourceRevisionID], _id)\\nRETURN IF(_allowed && NOT ISBLANK(_key) && HASONEVALUE(\'fm_review_measures\'[MetricKey]),\\n    CALCULATE(SUM(\'fm_review_facts\'[Amount]), TREATAS({_id}, \'fm_review_facts\'[SourceRevisionID]),\\n        TREATAS({_month}, \'fm_review_facts\'[Month]), TREATAS({_key}, \'fm_review_facts\'[ComponentKey])))\\n)", "formatString": "0.############", "displayFolder": "Forecast review\\\\Visual feed", "dynamic": false}, "FM UI Denominator": {"expression": "VAR _country = CALCULATE(SELECTEDVALUE(\'fm_review_reference\'[Country]), ALLSELECTED(\'fm_review_reference\'))\\nVAR _cycles = CALCULATETABLE(VALUES(\'fm_review_reference\'[DemandCycle]), ALLSELECTED(\'fm_review_reference\'))\\nVAR _selectedIDs = SELECTCOLUMNS(ALLSELECTED(\'fm_review_reference\'), \\"ID\\", \'fm_review_reference\'[SourceRevisionID])\\nVAR _expectedIDs = SELECTCOLUMNS(FILTER(ALL(\'fm_review_reference\'), \'fm_review_reference\'[Country] = _country && \'fm_review_reference\'[DemandCycle] IN _cycles), \\"ID\\", \'fm_review_reference\'[SourceRevisionID])\\nVAR _allMetrics = CALCULATE(DISTINCTCOUNT(\'fm_review_measures\'[MetricKey]), ALLSELECTED(\'fm_review_measures\')) = COUNTROWS(ALL(\'fm_review_measures\'))\\nVAR _cycleComplete = NOT ISBLANK(_country) && ISEMPTY(EXCEPT(_expectedIDs, _selectedIDs)) && _allMetrics\\nVAR _scopeAllowed = _cycleComplete\\nRETURN IF(_scopeAllowed,\\nVAR _id = SELECTEDVALUE(\'fm_review_files\'[SourceRevisionID])\\nVAR _month = SELECTEDVALUE(\'fm_review_axis\'[Month])\\nVAR _key = SELECTEDVALUE(\'fm_review_measures\'[DenominatorKey])\\nVAR _allowed = CONTAINS(VALUES(\'fm_review_reference\'[SourceRevisionID]), \'fm_review_reference\'[SourceRevisionID], _id)\\nRETURN IF(_allowed && NOT ISBLANK(_key) && HASONEVALUE(\'fm_review_measures\'[MetricKey]),\\n    CALCULATE(SUM(\'fm_review_facts\'[Amount]), TREATAS({_id}, \'fm_review_facts\'[SourceRevisionID]),\\n        TREATAS({_month}, \'fm_review_facts\'[Month]), TREATAS({_key}, \'fm_review_facts\'[ComponentKey])))\\n)", "formatString": "0.############", "displayFolder": "Forecast review\\\\Visual feed", "dynamic": false}, "FM UI Numerator Missing": {"expression": "VAR _country = CALCULATE(SELECTEDVALUE(\'fm_review_reference\'[Country]), ALLSELECTED(\'fm_review_reference\'))\\nVAR _cycles = CALCULATETABLE(VALUES(\'fm_review_reference\'[DemandCycle]), ALLSELECTED(\'fm_review_reference\'))\\nVAR _selectedIDs = SELECTCOLUMNS(ALLSELECTED(\'fm_review_reference\'), \\"ID\\", \'fm_review_reference\'[SourceRevisionID])\\nVAR _expectedIDs = SELECTCOLUMNS(FILTER(ALL(\'fm_review_reference\'), \'fm_review_reference\'[Country] = _country && \'fm_review_reference\'[DemandCycle] IN _cycles), \\"ID\\", \'fm_review_reference\'[SourceRevisionID])\\nVAR _allMetrics = CALCULATE(DISTINCTCOUNT(\'fm_review_measures\'[MetricKey]), ALLSELECTED(\'fm_review_measures\')) = COUNTROWS(ALL(\'fm_review_measures\'))\\nVAR _cycleComplete = NOT ISBLANK(_country) && ISEMPTY(EXCEPT(_expectedIDs, _selectedIDs)) && _allMetrics\\nVAR _scopeAllowed = _cycleComplete\\nRETURN IF(_scopeAllowed,\\nVAR _id = SELECTEDVALUE(\'fm_review_files\'[SourceRevisionID])\\nVAR _month = SELECTEDVALUE(\'fm_review_axis\'[Month])\\nVAR _key = SELECTEDVALUE(\'fm_review_measures\'[NumeratorKey])\\nVAR _allowed = CONTAINS(VALUES(\'fm_review_reference\'[SourceRevisionID]), \'fm_review_reference\'[SourceRevisionID], _id)\\nRETURN IF(_allowed && NOT ISBLANK(_key) && HASONEVALUE(\'fm_review_measures\'[MetricKey]),\\n    CALCULATE(SUM(\'fm_review_facts\'[MissingCells]), TREATAS({_id}, \'fm_review_facts\'[SourceRevisionID]),\\n        TREATAS({_month}, \'fm_review_facts\'[Month]), TREATAS({_key}, \'fm_review_facts\'[ComponentKey])))\\n)", "formatString": "0.############", "displayFolder": "Forecast review\\\\Visual feed", "dynamic": false}, "FM UI Denominator Missing": {"expression": "VAR _country = CALCULATE(SELECTEDVALUE(\'fm_review_reference\'[Country]), ALLSELECTED(\'fm_review_reference\'))\\nVAR _cycles = CALCULATETABLE(VALUES(\'fm_review_reference\'[DemandCycle]), ALLSELECTED(\'fm_review_reference\'))\\nVAR _selectedIDs = SELECTCOLUMNS(ALLSELECTED(\'fm_review_reference\'), \\"ID\\", \'fm_review_reference\'[SourceRevisionID])\\nVAR _expectedIDs = SELECTCOLUMNS(FILTER(ALL(\'fm_review_reference\'), \'fm_review_reference\'[Country] = _country && \'fm_review_reference\'[DemandCycle] IN _cycles), \\"ID\\", \'fm_review_reference\'[SourceRevisionID])\\nVAR _allMetrics = CALCULATE(DISTINCTCOUNT(\'fm_review_measures\'[MetricKey]), ALLSELECTED(\'fm_review_measures\')) = COUNTROWS(ALL(\'fm_review_measures\'))\\nVAR _cycleComplete = NOT ISBLANK(_country) && ISEMPTY(EXCEPT(_expectedIDs, _selectedIDs)) && _allMetrics\\nVAR _scopeAllowed = _cycleComplete\\nRETURN IF(_scopeAllowed,\\nVAR _id = SELECTEDVALUE(\'fm_review_files\'[SourceRevisionID])\\nVAR _month = SELECTEDVALUE(\'fm_review_axis\'[Month])\\nVAR _key = SELECTEDVALUE(\'fm_review_measures\'[DenominatorKey])\\nVAR _allowed = CONTAINS(VALUES(\'fm_review_reference\'[SourceRevisionID]), \'fm_review_reference\'[SourceRevisionID], _id)\\nRETURN IF(_allowed && NOT ISBLANK(_key) && HASONEVALUE(\'fm_review_measures\'[MetricKey]),\\n    CALCULATE(SUM(\'fm_review_facts\'[MissingCells]), TREATAS({_id}, \'fm_review_facts\'[SourceRevisionID]),\\n        TREATAS({_month}, \'fm_review_facts\'[Month]), TREATAS({_key}, \'fm_review_facts\'[ComponentKey])))\\n)", "formatString": "0.############", "displayFolder": "Forecast review\\\\Visual feed", "dynamic": false}, "FM UI File": {"expression": "VAR _scopeAllowed = CONTAINS(VALUES(\'fm_review_reference\'[SourceRevisionID]), \'fm_review_reference\'[SourceRevisionID], MINX(ALLSELECTED(\'fm_review_reference\'), \'fm_review_reference\'[SourceRevisionID])) && SELECTEDVALUE(\'fm_review_measures\'[MetricKey]) = MINX(ALLSELECTED(\'fm_review_measures\'), \'fm_review_measures\'[MetricKey])\\nRETURN IF(_scopeAllowed,\\nVAR _month = SELECTEDVALUE(\'fm_review_axis\'[Month])\\nVAR _first = MINX(ALL(\'fm_review_axis\'), \'fm_review_axis\'[Month])\\nRETURN IF(_month = _first, \\"{\\" & \\"\\"\\"id\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(COALESCE(SELECTEDVALUE(\'fm_review_files\'[SourceRevisionID]), \\"\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"fullLabel\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(COALESCE(SELECTEDVALUE(\'fm_review_files\'[FileLabel]), \\"\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"filename\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(COALESCE(SELECTEDVALUE(\'fm_review_files\'[SourceFileName]), \\"\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"path\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(COALESCE(SELECTEDVALUE(\'fm_review_files\'[SourceRelativePath]), \\"\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"affiliate\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(COALESCE(SELECTEDVALUE(\'fm_review_files\'[Country]), \\"\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"cycle\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(COALESCE(SELECTEDVALUE(\'fm_review_files\'[DemandCycle]), \\"\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"driver\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(COALESCE(SELECTEDVALUE(\'fm_review_files\'[GrowthDriver]), \\"\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"profile\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(COALESCE(SELECTEDVALUE(\'fm_review_files\'[SourceProfile]), \\"\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"status\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(COALESCE(SELECTEDVALUE(\'fm_review_files\'[PublicationStatus]), \\"\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"ims\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(FORMAT(SELECTEDVALUE(\'fm_review_files\'[IMSActualsThrough]), \\"yyyy-MM\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"ef\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(FORMAT(SELECTEDVALUE(\'fm_review_files\'[ExFactoryActualsThrough]), \\"yyyy-MM\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\"}\\")\\n)", "formatString": "", "displayFolder": "Forecast review\\\\Visual feed", "dynamic": false}, "FM UI Catalog": {"expression": "VAR _scopeAllowed = CONTAINS(VALUES(\'fm_review_reference\'[SourceRevisionID]), \'fm_review_reference\'[SourceRevisionID], MINX(ALLSELECTED(\'fm_review_reference\'), \'fm_review_reference\'[SourceRevisionID])) && SELECTEDVALUE(\'fm_review_measures\'[MetricKey]) = MINX(ALLSELECTED(\'fm_review_measures\'), \'fm_review_measures\'[MetricKey])\\nRETURN IF(_scopeAllowed,\\nVAR _firstMonth = MINX(ALL(\'fm_review_axis\'), \'fm_review_axis\'[Month])\\nVAR _firstFile = MINX(ALL(\'fm_review_files\'), \'fm_review_files\'[SourceRevisionID])\\nVAR _first = SELECTEDVALUE(\'fm_review_axis\'[Month]) = _firstMonth && SELECTEDVALUE(\'fm_review_files\'[SourceRevisionID]) = _firstFile\\n\\nRETURN IF(_first, \\"[\\" & CONCATENATEX(ALL(\'fm_review_measures\'), \\"{\\" & \\"\\"\\"id\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(\'fm_review_measures\'[MetricKey], UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"family\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(\'fm_review_measures\'[Section], UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"label\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(\'fm_review_measures\'[Measure], UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"unit\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(\'fm_review_measures\'[Unit], UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"aggregation\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(\'fm_review_measures\'[Aggregation], UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"cutoffFamily\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(\'fm_review_measures\'[CutoffFamily], UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"numeratorKey\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(\'fm_review_measures\'[NumeratorKey], UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"denominatorKey\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(\'fm_review_measures\'[DenominatorKey], UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"availabilityIndex\\"\\":\\" & FORMAT(RANKX(ALL(\'fm_review_measures\'), \'fm_review_measures\'[MetricKey], , ASC, DENSE) - 1, \\"0\\") & \\"}\\", \\",\\", \'fm_review_measures\'[SortIndex], ASC) & \\"]\\")\\n)", "formatString": "", "displayFolder": "Forecast review\\\\Visual feed", "dynamic": false}, "FM UI Context": {"expression": "VAR _country = CALCULATE(SELECTEDVALUE(\'fm_review_reference\'[Country]), ALLSELECTED(\'fm_review_reference\'))\\nVAR _cycles = CALCULATETABLE(VALUES(\'fm_review_reference\'[DemandCycle]), ALLSELECTED(\'fm_review_reference\'))\\nVAR _selectedIDs = SELECTCOLUMNS(ALLSELECTED(\'fm_review_reference\'), \\"ID\\", \'fm_review_reference\'[SourceRevisionID])\\nVAR _expectedIDs = SELECTCOLUMNS(FILTER(ALL(\'fm_review_reference\'), \'fm_review_reference\'[Country] = _country && \'fm_review_reference\'[DemandCycle] IN _cycles), \\"ID\\", \'fm_review_reference\'[SourceRevisionID])\\nVAR _allMetrics = CALCULATE(DISTINCTCOUNT(\'fm_review_measures\'[MetricKey]), ALLSELECTED(\'fm_review_measures\')) = COUNTROWS(ALL(\'fm_review_measures\'))\\nVAR _cycleComplete = NOT ISBLANK(_country) && ISEMPTY(EXCEPT(_expectedIDs, _selectedIDs)) && _allMetrics\\nVAR _scopeAllowed = CONTAINS(VALUES(\'fm_review_reference\'[SourceRevisionID]), \'fm_review_reference\'[SourceRevisionID], MINX(ALLSELECTED(\'fm_review_reference\'), \'fm_review_reference\'[SourceRevisionID])) && SELECTEDVALUE(\'fm_review_measures\'[MetricKey]) = MINX(ALLSELECTED(\'fm_review_measures\'), \'fm_review_measures\'[MetricKey])\\nRETURN IF(_scopeAllowed,\\nVAR _firstMonth = MINX(ALL(\'fm_review_axis\'), \'fm_review_axis\'[Month])\\nVAR _firstFile = MINX(ALL(\'fm_review_files\'), \'fm_review_files\'[SourceRevisionID])\\nVAR _first = SELECTEDVALUE(\'fm_review_axis\'[Month]) = _firstMonth && SELECTEDVALUE(\'fm_review_files\'[SourceRevisionID]) = _firstFile\\n\\nRETURN IF(_first, \\"{\\" & \\"\\"\\"affiliate\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(CALCULATE(SELECTEDVALUE(\'fm_review_reference\'[Country], \\"\\"), ALLSELECTED(\'fm_review_reference\')), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"profile\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(CALCULATE(SELECTEDVALUE(\'fm_review_reference\'[SourceProfile], \\"\\"), ALLSELECTED(\'fm_review_reference\')), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"metric\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(CALCULATE(SELECTEDVALUE(\'fm_review_measures\'[MetricKey], \\"\\"), ALLSELECTED(\'fm_review_measures\')), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"channel\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(IF(ISFILTERED(\'fm_review_facts\'[Channel]), SELECTEDVALUE(\'fm_review_facts\'[Channel], \\"all\\"), \\"all\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"subchannel\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(IF(ISFILTERED(\'fm_review_facts\'[SubChannel]), SELECTEDVALUE(\'fm_review_facts\'[SubChannel], \\"all\\"), \\"all\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"sector\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(IF(ISFILTERED(\'fm_review_facts\'[Sector]), SELECTEDVALUE(\'fm_review_facts\'[Sector], \\"all\\"), \\"all\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"schema\\"\\":\\" & \\"5\\" & \\",\\" & \\"\\"\\"complete\\"\\":\\" & IF(_cycleComplete, \\"true\\", \\"false\\") & \\",\\" & \\"\\"\\"cycle\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(CALCULATE(SELECTEDVALUE(\'fm_review_reference\'[DemandCycle], \\"\\"), ALLSELECTED(\'fm_review_reference\')), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"driver\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(CALCULATE(SELECTEDVALUE(\'fm_review_reference\'[GrowthDriver], \\"\\"), ALLSELECTED(\'fm_review_reference\')), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"cycles\\"\\":\\" & \\"[\\" & CONCATENATEX(SUMMARIZE(ALLSELECTED(\'fm_review_reference\'), \'fm_review_reference\'[DemandCycle]), UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(\'fm_review_reference\'[DemandCycle], UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34), \\",\\", \'fm_review_reference\'[DemandCycle], ASC) & \\"]\\" & \\",\\" & \\"\\"\\"drivers\\"\\":\\" & \\"[\\" & CONCATENATEX(SUMMARIZE(ALLSELECTED(\'fm_review_reference\'), \'fm_review_reference\'[GrowthDriver]), UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(\'fm_review_reference\'[GrowthDriver], UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34), \\",\\", \'fm_review_reference\'[GrowthDriver], ASC) & \\"]\\" & \\"}\\")\\n)", "formatString": "", "displayFolder": "Forecast review\\\\Visual feed", "dynamic": false}, "FM UI Dimensions": {"expression": "BLANK()", "formatString": "", "displayFolder": "Forecast review\\\\Visual feed", "dynamic": false}, "FM UI Bounds": {"expression": "VAR _scopeAllowed = CONTAINS(VALUES(\'fm_review_reference\'[SourceRevisionID]), \'fm_review_reference\'[SourceRevisionID], MINX(ALLSELECTED(\'fm_review_reference\'), \'fm_review_reference\'[SourceRevisionID])) && SELECTEDVALUE(\'fm_review_measures\'[MetricKey]) = MINX(ALLSELECTED(\'fm_review_measures\'), \'fm_review_measures\'[MetricKey])\\nRETURN IF(_scopeAllowed,\\nVAR _firstMonth = MINX(ALL(\'fm_review_axis\'), \'fm_review_axis\'[Month])\\nVAR _firstFile = MINX(ALL(\'fm_review_files\'), \'fm_review_files\'[SourceRevisionID])\\nVAR _first = SELECTEDVALUE(\'fm_review_axis\'[Month]) = _firstMonth && SELECTEDVALUE(\'fm_review_files\'[SourceRevisionID]) = _firstFile\\nRETURN IF(_first, \\"{\\" & \\"\\"\\"first\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(FORMAT(MINX(ALL(\'fm_review_axis\'), \'fm_review_axis\'[Month]), \\"yyyy-MM\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\",\\" & \\"\\"\\"last\\"\\":\\" & UNICHAR(34) & SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(FORMAT(MAXX(ALL(\'fm_review_axis\'), \'fm_review_axis\'[Month]), \\"yyyy-MM\\"), UNICHAR(92), UNICHAR(92) & UNICHAR(92)), UNICHAR(34), UNICHAR(92) & UNICHAR(34)), UNICHAR(13), \\"\\"), UNICHAR(10), UNICHAR(92) & \\"n\\"), UNICHAR(9), UNICHAR(92) & \\"t\\") & UNICHAR(34) & \\"}\\")\\n)", "formatString": "", "displayFolder": "Forecast review\\\\Visual feed", "dynamic": false}}')
result = main(EXPECTED, UPDATES)
